In [0]:
# %pip install pypdf

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Databricks Notebook: 01_medallion_etl
import io
import json
import hashlib
import requests
import pandas as pd
from pypdf import PdfReader
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

spark = SparkSession.builder.getOrCreate()

CATALOG = "rag"
SCHEMA = "wb_operations"
VOLUME_NAME = "raw_pad_pdfs"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

# ==============================================================================
# 1. BRONZE LAYER: Raw Append-Only Ingestion
# ==============================================================================

def ingest_bronze_structured(rows: int = 500):
    """Fetches raw JSON payload from World Bank Projects API and stores it in Bronze."""
    url = f"https://search.worldbank.org/api/v2/projects?format=json&rows={rows}"
    response = requests.get(url, timeout=30).json()
    projects = response.get("projects", {})
    
    raw_records = [{"project_id": pid, "raw_payload": json.dumps(data)} 
                   for pid, data in projects.items() if isinstance(data, dict)]
    
    df_raw = pd.DataFrame(raw_records)
    (spark.createDataFrame(df_raw)
     .withColumn("ingested_at", F.current_timestamp())
     .write.format("delta")
     .mode("append")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_wb_projects"))
    print("Bronze structured projects ingested.")

def ingest_bronze_unstructured(max_docs: int = 15):
    """Downloads raw PAD PDFs into a Unity Catalog Volume and records metadata in Bronze."""
    docs_api_url = f"https://search.worldbank.org/api/v2/wds?format=json&docty_exact=Project+Appraisal+Document&rows={max_docs}"
    res = requests.get(docs_api_url, timeout=30).json()
    documents = res.get("documents", {})
    
    metadata_records = []
    for doc_id, meta in documents.items():
        if not isinstance(meta, dict):
            continue
        pdf_url = meta.get("pdfurl")
        project_id = meta.get("proj_id", "UNKNOWN")
        doc_title = meta.get("display_title", "PAD Document")
        
        if pdf_url:
            try:
                pdf_res = requests.get(pdf_url, timeout=40)
                file_name = f"{project_id}_{doc_id}.pdf"
                file_path = f"{VOLUME_PATH}/{file_name}"
                
                with open(file_path, "wb") as f:
                    f.write(pdf_res.content)
                    
                metadata_records.append({
                    "doc_id": doc_id,
                    "project_id": project_id,
                    "document_title": doc_title,
                    "file_path": file_path,
                    "source_url": pdf_url
                })
            except Exception as e:
                print(f"Failed downloading {pdf_url}: {e}")

    if metadata_records:
        df_docs = pd.DataFrame(metadata_records)
        (spark.createDataFrame(df_docs)
         .withColumn("ingested_at", F.current_timestamp())
         .write.format("delta")
         .mode("append")
         .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_wb_documents"))
        print("Bronze unstructured PDFs downloaded and cataloged.")

# ==============================================================================
# 2. SILVER LAYER: Parsing, Cleaning, and Semantic Chunking
# ==============================================================================

def process_silver_structured():
    """Parses JSON payloads into a strongly typed, deduplicated Silver Delta table."""
    bronze_df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_wb_projects")
    
    # Extract fields with PySpark JSON functions
    silver_df = bronze_df.select(
        F.col("project_id"),
        F.get_json_object("raw_payload", "$.project_name").alias("project_name"),
        F.get_json_object("raw_payload", "$.countryname").alias("country"),
        F.get_json_object("raw_payload", "$.regionname").alias("region"),
        F.regexp_replace(F.get_json_object("raw_payload", "$.totalamt"), ",", "").cast(DoubleType()).alias("total_commitment_usd"),
        F.regexp_replace(F.get_json_object("raw_payload", "$.ibrdcommamt"), ",", "").cast(DoubleType()).alias("ibrd_commitment_usd"),
        F.regexp_replace(F.get_json_object("raw_payload", "$.idacommamt"), ",", "").cast(DoubleType()).alias("ida_commitment_usd"),
        F.get_json_object("raw_payload", "$.status").alias("status"),
        F.to_date(F.get_json_object("raw_payload", "$.boardapprovaldate")).alias("approval_date"),
        F.coalesce(
            F.get_json_object("raw_payload", "$.sector1.Name"),
            F.get_json_object("raw_payload", "$.sector1")
        ).alias("primary_sector")
    ).dropDuplicates(["project_id"])
    
    silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_wb_projects")
    print("Silver structured projects processed.")

def process_silver_unstructured():
    """Extracts text from Volume PDFs, segments into chunks with overlap, and computes hashes."""
    bronze_docs = spark.table(f"{CATALOG}.{SCHEMA}.bronze_wb_documents").toPandas()
    chunks = []
    
    for _, row in bronze_docs.iterrows():
        try:
            reader = PdfReader(row["file_path"])
            # Process main analytical body (typically pages 1-40 in PADs)
            for page_idx, page in enumerate(reader.pages[:40]):
                text = page.extract_text()
                if not text or len(text.strip()) < 80:
                    continue
                
                # Chunk with a 1000-character window and 150-character overlap
                step = 850
                for start in range(0, len(text), step):
                    chunk_str = text[start:start + 1000].strip()
                    if len(chunk_str) > 100:
                        chunk_id = hashlib.md5(f"{row['doc_id']}_{page_idx}_{start}".encode()).hexdigest()
                        chunks.append({
                            "chunk_id": chunk_id,
                            "project_id": row["project_id"],
                            "document_title": row["document_title"],
                            "page_number": page_idx + 1,
                            "source_url": row["source_url"],
                            "chunk_text": chunk_str
                        })
        except Exception as e:
            print(f"Error parsing PDF {row['file_path']}: {e}")
            
    if chunks:
        spark.createDataFrame(pd.DataFrame(chunks)) \
             .dropDuplicates(["chunk_id"]) \
             .write.format("delta") \
             .mode("overwrite") \
             .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_wb_pad_chunks")
        print("Silver unstructured chunks generated.")

# ==============================================================================
# 3. GOLD LAYER: Optimized Aggregations & Vector Sync Source Table
# ==============================================================================

def process_gold_layer():
    """Builds optimized analytics tables and enables CDF for Vector Search."""
    # 3A. Gold Projects (Filtered, verified schema)
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_wb_projects AS
        SELECT 
            project_id,
            project_name,
            country,
            region,
            COALESCE(total_commitment_usd, 0.0) AS total_commitment_usd,
            COALESCE(ibrd_commitment_usd, 0.0) AS ibrd_commitment_usd,
            COALESCE(ida_commitment_usd, 0.0) AS ida_commitment_usd,
            status,
            approval_date,
            COALESCE(primary_sector, 'Other') AS primary_sector
        FROM {CATALOG}.{SCHEMA}.silver_wb_projects
    """)

    # 3B. Gold PAD Chunks with CDF Enabled
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.gold_wb_pad_chunks AS
        SELECT 
            chunk_id,
            project_id,
            document_title,
            page_number,
            source_url,
            chunk_text
        FROM {CATALOG}.{SCHEMA}.silver_wb_pad_chunks
    """)
    
    # Crucial: Enable CDF for Databricks Vector Search delta sync
    spark.sql(f"""
        ALTER TABLE {CATALOG}.{SCHEMA}.gold_wb_pad_chunks 
        SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)
    print("Gold tables materialized with Change Data Feed enabled.")

# Run End-to-End ETL
ingest_bronze_structured()
ingest_bronze_unstructured()
process_silver_structured()
process_silver_unstructured()
process_gold_layer()

Bronze structured projects ingested.
Bronze unstructured PDFs downloaded and cataloged.
Silver structured projects processed.
Silver unstructured chunks generated.
Gold tables materialized with Change Data Feed enabled.
